In [ ]:
import subprocess, time, os, socket, tempfile
from IPython.display import display, HTML, IFrame

port  = 3838
path  = '/home/jovyan/tools/wordfinder'
name  = 'WordFinder'
emoji = '\ud83d\udd0d'

log_file = os.path.join(tempfile.gettempdir(), 'shiny_wordfinder.log')

# Show loading message
display(HTML(f'''
<div style="font-family:sans-serif;padding:20px;background:#f4f0f8;
            border-left:4px solid #51247a;border-radius:8px;margin-bottom:16px;">
  <span style="font-size:1.4rem;">{emoji}</span>
  <b style="font-size:1.1rem;color:#51247a;"> {name}</b><br>
  <span style="color:#666;font-size:.9rem;">
    Keyword-in-context concordancing &mdash; starting, please wait...
  </span>
</div>
'''))

# Kill any stale process on this port
subprocess.run(['fuser', '-k', f'{port}/tcp'], capture_output=True, check=False)
time.sleep(1)

# Start Shiny, log output for debugging
with open(log_file, 'w') as log:
    proc = subprocess.Popen(
        ['R', '--vanilla', '-e',
         f"shiny::runApp('{path}', port={port}, host='0.0.0.0', launch.browser=FALSE)"],
        stdout=log,
        stderr=subprocess.STDOUT
    )

# Poll until port is open or process exits
ready = False
for i in range(120):
    time.sleep(1)
    if proc.poll() is not None:
        # Process exited — show error log
        try:
            with open(log_file) as f:
                log_content = f.read()[-3000:]
        except:
            log_content = '(no log available)'
        display(HTML(f'''
        <div style="padding:16px;background:#fff0f0;border-left:4px solid #e74c3c;
                    border-radius:6px;font-family:sans-serif;">
          <b style="color:#c0392b;">&#x274C; {name} failed to start.</b><br><br>
          <details><summary style="cursor:pointer;color:#c0392b;font-weight:600;">Show error log</summary>
          <pre style="background:#fff;padding:10px;border-radius:4px;font-size:.78rem;
                      overflow-x:auto;white-space:pre-wrap;max-height:300px;">{log_content}</pre>
          </details>
        </div>
        '''))
        break
    try:
        s = socket.create_connection(('localhost', port), timeout=1)
        s.close()
        ready = True
        break
    except:
        pass

if ready:
    # Build the proxy URL — this is how Voila + jupyter-server-proxy work together
    base_url = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
    tool_url  = f'{base_url}proxy/{port}/'

    display(HTML(f'''
    <div style="font-family:sans-serif;padding:12px 16px;background:#eafaf1;
                border-left:4px solid #27ae60;border-radius:6px;margin-bottom:12px;">
      &#x2705; <b>{name} is ready.</b>
      <a href="{tool_url}" target="_blank"
         style="margin-left:16px;padding:6px 18px;background:#51247a;color:white;
                border-radius:6px;text-decoration:none;font-weight:600;font-size:.9rem;">
        {emoji} Open in new tab
      </a>
      <span style="color:#888;font-size:.82rem;margin-left:10px;">
        Or use the embedded view below.
      </span>
    </div>
    '''))

    # Embed the Shiny app directly in the page
    display(IFrame(src=tool_url, width='100%', height='850px'))
